# Service HMM / Viterbi — Kernel Gateway

Exposition REST du decodage **Viterbi** (et de l'algorithme de l'avant) du carnet phare [05-HMM-Viterbi.ipynb](../05-HMM-Viterbi.ipynb) — raisonnement probabiliste sur la duree, ch. 14 d'AIMA.

Feuille de service du **pli 1** de l'Epic AIMA-Gateway #19571. Le carnet pedagogique n'est pas modifie : ce dossier `gateway/` ajoute un carnet de service a cote, sur le meme motif que les pilotes `GameTheory/gateway/`, `Probas/gateway/` et `Z3-API/gateway/`.

**Contrat** : `POST /nlp/viterbi` — corps = modele HMM (etats, observations, `pi`, `A`, `B`), reponse = chemin optimal de Viterbi + log-probabilites. Le detail du contrat et le verdict SOTA sont dans le [README](README.md) de ce dossier.

In [1]:
# Preparation : charger le coeur du service (testable hors kernel) et definir
# l'exemple meteo du carnet phare.
import importlib.util
from pathlib import Path

# Le coeur vit a cote du carnet de service (le kernel est demarre dans ce
# dossier, que ce soit par kernel_gateway ou par papermill/nbconvert).
_core_path = Path("hmm_viterbi_core.py")
_spec = importlib.util.spec_from_file_location("hmm_viterbi_core", _core_path)
core = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(core)
run = core.run

# Exemple meteo classique (celui du carnet phare) : le temps et l'activite.
weather = {
    "states": ["EnSoleil", "Pluie"],
    "observations": ["promenade", "magasin", "nettoyage"],
    "pi": [0.6, 0.4],
    "A": [[0.7, 0.3], [0.4, 0.6]],
    "B": {
        "EnSoleil": {"promenade": 0.7, "magasin": 0.4, "nettoyage": 0.1},
        "Pluie": {"promenade": 0.1, "magasin": 0.4, "nettoyage": 0.6},
    },
}
print("Coeur charge :", _core_path.name)
print("Etats       :", weather["states"])
print("Observations:", weather["observations"])

Coeur charge : hmm_viterbi_core.py
Etats       : ['EnSoleil', 'Pluie']
Observations: ['promenade', 'magasin', 'nettoyage']


La sortie confirme le chargement du coeur `hmm_viterbi_core.py` et fixe le modèle : 2 états (`EnSoleil`, `Pluie`) et 3 observations (`promenade`, `magasin`, `nettoyage`).

In [2]:
# POST /nlp/viterbi
#
# Point d'entree du service. En mode gateway, kernel_gateway route les POST
# sur /nlp/viterbi vers `http_server` ; le corps JSON est passe a `run()` du
# coeur et la reponse est serialisee. Le carnet s'exécute aussi hors gateway
# (papermill/nbconvert) : `kernel_gateway` n'est alors pas importe.
import json

try:  # mode gateway : le type HTTPRequest existe
    from kernel_gateway import HTTPRequest  # noqa: F401
    _GATEWAY = True
except ImportError:  # mode hors gateway (papermill/nbconvert)
    _GATEWAY = False


def http_server(request, kernel_session):
    """Route `POST /nlp/viterbi` : `run()` du coeur, reponse JSON au contrat."""
    try:
        body = json.loads(request.body)
    except Exception as exc:
        payload = {"status": "invalid", "error": f"corps JSON invalide: {exc}"}
        return 200, json.dumps(payload, ensure_ascii=False).encode("utf-8")
    return 200, json.dumps(run(body), ensure_ascii=False).encode("utf-8")


print("Mode gateway :", "oui" if _GATEWAY else "non (demo hors gateway)")

Mode gateway :

non (demo hors gateway)

Hors gateway, la cellule affiche `non (demo hors gateway)` : `kernel_gateway` n'est pas importable, et `http_server` reste simplement défini sans être appelé.

In [3]:
# Demonstration : decodage de Viterbi sur l'exemple meteo (sortie reelle).
import json

response = run(weather)
print(json.dumps(response, ensure_ascii=False, indent=2))

{
  "status": "ok",
  "best_path": [
    "EnSoleil",
    "EnSoleil",
    "Pluie"
  ],
  "best_path_log_prob": -3.855264671609537,
  "observation_log_prob": -2.900422093749666,
  "n_states": 2,
  "n_observations": 3
}

In [4]:
# Demonstration : entree hors contrat -> `status: invalid`, aucune exception.
import json

bad = dict(weather)
bad["pi"] = [0.6, 0.4, 0.0]  # 3 probabilites pour 2 etats
print(json.dumps(run(bad), ensure_ascii=False, indent=2))

{
  "status": "invalid",
  "error": "pi: liste de 2 probabilites attendue"
}

## Lecture du resultat

Le decodage de Viterbi retourne le **chemin d'etats le plus probable** (calcule en espace des logarithmes pour la stabilite numerique) et la log-probabilite de ce chemin. `observation_log_prob` est la log-probabilite marginale des observations (algorithme de l'avant) : elle **majorise** celle du meilleur chemin, car elle somme sur toutes les trajectoires.

**Verdict SOTA : SOTA-OK.** Le service expose le decodage Viterbi / avant du carnet phare — l'organe natif de la serie NLP pour cet algorithme (construit a la main dans le carnet, ch. 14 AIMA). La reprise est **auto-contenue et fidele** au carnet : meme algorithme, en espace des log. Le carnet enseigne precisement l'algorithme que le service expose, il n'y a pas d'outil externe que le service pretendrait utiliser sans le faire. (`hmmlearn` existe comme alternative externe ; le point d'un service de cette serie pedagogique est d'exposer l'implementation de reference du carnet, pas de la deleger a une bibliotheque.)